# DTDLINK — Stage 1 raw-data audit

Chỉ audit, chưa cleaning/training/target mapping. Code đầy đủ: `src/dtdlink_ml/data_audit.py`. Run All chạy lại bằng môi trường dự án. URL giữ dạng text, không truy cập. Báo cáo: `reports/audit/stage-1-data-audit.md`.

In [1]:
from pathlib import Path
import sys, json
root = Path.cwd()
if not (root / 'src/dtdlink_ml/data_audit.py').exists():
    root = root.parent
assert (root / 'src/dtdlink_ml/data_audit.py').exists()
sys.path.insert(0, str(root / 'src'))
from dtdlink_ml.data_audit import run_audit
previous = json.loads((root / 'reports/audit/summary.json').read_text())
summary = run_audit()
assert {k:v for k,v in previous.items() if k != 'audit_time_utc'} == {k:v for k,v in summary.items() if k != 'audit_time_utc'}
print('Rerun matches previous audit; source hashes unchanged.')

Rerun matches previous audit; source hashes unchanged.


## Số liệu tổng hợp
Kiểu string được chỉ định để giữ nguyên văn bản; CSV tự nó không có schema kiểu dữ liệu.

In [2]:
for key in ['shape','schema','labels','missingness','duplicates','conflicts','by_class','diagnostic_host','verification']:
    print(key)
    print(json.dumps(summary[key], ensure_ascii=False, indent=2))

shape
{
  "rows": 651191,
  "columns": 2
}
schema
[
  {
    "column": "url",
    "pandas_dtype": "string",
    "semantic_type": "URL text"
  },
  {
    "column": "type",
    "pandas_dtype": "string",
    "semantic_type": "categorical label"
  }
]
labels
[
  {
    "label": "benign",
    "rows": 428103,
    "percent": 65.74154126823007
  },
  {
    "label": "defacement",
    "rows": 96457,
    "percent": 14.812397591490054
  },
  {
    "label": "malware",
    "rows": 32520,
    "percent": 4.993926513112128
  },
  {
    "label": "phishing",
    "rows": 94111,
    "percent": 14.452134627167759
  }
]
missingness
{
  "url": {
    "null": 0,
    "empty_string": 0,
    "whitespace_only_nonempty": 0,
    "na_like_token": 0,
    "leading_or_trailing_whitespace": 7
  },
  "type": {
    "null": 0,
    "empty_string": 0,
    "whitespace_only_nonempty": 0,
    "na_like_token": 0,
    "leading_or_trailing_whitespace": 0
  }
}
duplicates
{
  "unique_complete_rows": 641125,
  "exact_row_excess": 10066,

## Các phép kiểm tra chính có thể đọc trực tiếp
Không gán lại URL hoặc label; duplicated chỉ đếm, không loại hàng.

In [3]:
import pandas as pd
df = pd.read_csv(root / 'data/raw/malicious_phish.csv', dtype='string', keep_default_na=False, na_filter=False, skip_blank_lines=False)
assert list(df.shape) == [summary['shape']['rows'], summary['shape']['columns']]
assert df['type'].value_counts().to_dict() == {r['label']:r['rows'] for r in summary['labels']}
assert int(df.duplicated().sum()) == summary['duplicates']['exact_row_excess']
assert int(df['url'].duplicated().sum()) == summary['duplicates']['url_excess']
labels_per_url = df.groupby('url')['type'].nunique()
assert int((labels_per_url > 1).sum()) == summary['conflicts']['unique_urls']
print('Shape, labels, duplicates and conflicts agree.')

Shape, labels, duplicates and conflicts agree.


## Ví dụ và bằng chứng xung đột
Số bản ghi là thứ tự sau header, không phải số dòng vật lý CSV. Ví dụ minh họa Q1/median/Q3 về độ dài, không xác thực nhãn. JSON hiển thị dạng text.

In [4]:
for name in ['examples.jsonl', 'conflict_records.jsonl']:
    print(name)
    print((root / 'reports/audit' / name).read_text())

examples.jsonl
{"record_number": 3, "label": "benign", "length_quantile": 0.25, "url": "bopsecrets.org/rexroth/cr/1.htm", "length": 31}
{"record_number": 8, "label": "benign", "length_quantile": 0.5, "url": "yourbittorrent.com/?q=anthony-hamilton-soulife", "length": 46}
{"record_number": 221, "label": "benign", "length_quantile": 0.75, "url": "elliscountydivorcelawyer.com/ellis-county-post-divorce-modification.htm", "length": 71}
{"record_number": 69, "label": "defacement", "length_quantile": 0.25, "url": "http://www.musimagen.com/lista_socios.php?letra=\\303%91", "length": 55}
{"record_number": 138, "label": "defacement", "length_quantile": 0.5, "url": "http://www.paoarapanui.cl/index.php?option=com_contact&view=contact&id=1&Itemid=6", "length": 81}
{"record_number": 1616, "label": "defacement", "length_quantile": 0.75, "url": "http://gbcmonroe.org/index.php?option=com_jevents&task=week.listevents&year=2013&month=02&day=3&Itemid=67", "length": 105}
{"record_number": 102868, "label": "

## Quyết định còn mở
Đọc mục 10 của báo cáo: A benign/phishing; B benign/potentially dangerous; C bốn lớp. Đề xuất A để bám sát phạm vi DTDLINK, nhưng phải chờ người dùng xác nhận. Stage 1 chưa hoàn thành gate target. Không dùng kết quả audit này để khẳng định URL đang an toàn hoặc nhãn nguồn chắc chắn đúng.